# Pandas — selekcja i filtrowanie

**Programowanie w Pythonie II** | Wykład 6
**Politechnika Opolska** | Analityka danych w biznesie

---

## Co dzisiaj?

Na W05 nauczyliśmy się tworzyć Series i DataFrame, wczytywać CSV i eksplorować dane
(`info()`, `describe()`, `value_counts()`). Wiemy **co** mamy w danych.

Dzisiaj przechodzimy od pytania *„co mam?"* do pytania *„pokaż mi dokładnie TO"*.
Np.: *„Pokaż rachunki powyżej 30 dolarów, z soboty, od niepalących, posortowane malejąco"*.

```mermaid
graph TD
    A["W05: Pandas intro"] --> B["Series, DataFrame"]
    A --> C["Eksploracja — info, describe"]
    A --> D["Wiem CO mam w danych"]
    D --> E["W06: selekcja i filtrowanie"]
    E --> F["iloc — po pozycji (numerze)"]
    E --> G["loc — po etykiecie/warunku"]
    E --> H["Warunki logiczne: &, |, isin, between"]
    E --> I["Sortowanie i rankingi"]
    E --> J["Segmentacja klientów"]
    E -.-> K["W07: czyszczenie danych"]
```

### Po tym wykładzie potrafisz:

1. **Rozróżniać** `loc` (etykiety) i `iloc` (pozycje) i stosować je poprawnie (Bloom 2-3)
2. **Filtrować** dane z warunkami logicznymi: AND, OR, negacja, isin, between, query (Bloom 3)
3. **Sortować** DataFrame i tworzyć rankingi: sort_values, nlargest, nsmallest (Bloom 3)
4. **Analizować** dane biznesowe odpowiadając na precyzyjne pytania (Bloom 4)

**Dokumentacja:** [Indexing and Selecting](https://pandas.pydata.org/docs/user_guide/indexing.html) |
[Boolean Indexing](https://pandas.pydata.org/docs/user_guide/indexing.html#boolean-indexing) |
[sort_values](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.sort_values.html)

In [ ]:
import pandas as pd
import numpy as np

# Dataset tips — rachunki i napiwki w restauracji (znamy go z W05)
tips = pd.read_csv('https://raw.githubusercontent.com/mwaskom/seaborn-data/master/tips.csv')
print(f"Dataset tips: {tips.shape[0]} wierszy, {tips.shape[1]} kolumn")
tips.head()

---

## 1. iloc — selekcja po pozycji (numerze)

`iloc` = **i**nteger **loc**ation — wybierasz wiersze i kolumny **po numerze pozycji**.
Działa identycznie jak indeksowanie w NumPy: `tablica[wiersz, kolumna]`.

### Analogia do Excela

| Co chcesz | Excel | Pandas iloc |
|-----------|-------|-------------|
| Komórka A1 | Kliknij A1 | `df.iloc[0, 0]` |
| Wiersze 1-5 | Zaznacz wiersze 1-5 | `df.iloc[0:5]` |
| Kolumny A-C | Zaznacz kolumny A-C | `df.iloc[:, 0:3]` |
| Co 5. wiersz | Filtr zaawansowany | `df.iloc[::5]` |

**Składnia:** `df.iloc[wiersze, kolumny]`
- Wiersze i kolumny to **numery** (od 0!)
- Slicing: `0:3` = pozycje 0, 1, 2 (stop **nie jest** włączony — jak w NumPy)

In [ ]:
# iloc — pierwszy wiersz
print("Wiersz 0 (pierwszy):")
print(tips.iloc[0])
print(f"\nTyp wyniku: {type(tips.iloc[0]).__name__}")  # Series!

In [ ]:
# iloc — slice wierszy i kolumn
print("Wiersze 0-2, kolumny 0-2:")
print(tips.iloc[0:3, 0:3])
print()
print("Przypomnienie: 0:3 = pozycje 0, 1, 2 (stop nie włączony)")

In [ ]:
# iloc — ostatni wiersz, co piąty wiersz
print("Ostatni wiersz:")
print(tips.iloc[-1])

print("\nCo piąty wiersz (kolumny 0-1):")
print(tips.iloc[::5, [0, 1]].head())

In [ ]:
# iloc — konkretne pozycje
print("Wiersze 10, 20, 30 — kolumny 0 i 4:")
print(tips.iloc[[10, 20, 30], [0, 4]])
print()
print("Lista pozycji: [10, 20, 30] — dowolne wiersze")

In [ ]:
# iloc vs loc — porównanie na tym samym DataFrame
print("=== iloc (po pozycji) ===")
print(tips.iloc[0:3])    # wiersze 0, 1, 2
print()
print("=== loc (po etykiecie — tu indeks numeryczny, więc wygląda podobnie) ===")
print(tips.loc[0:2])     # wiersze 0, 1, 2 WŁĄCZNIE!
print()
print("UWAGA: iloc[0:3] → 3 wiersze (0,1,2). loc[0:2] → 3 wiersze (0,1,2)!")
print("loc WŁĄCZA koniec zakresu, iloc NIE. To częsta pułapka.")

### Kiedy używać iloc?

- Potrzebujesz **n-ty wiersz** (np. pierwszy, ostatni, co dziesiąty)
- Nie znasz nazw kolumn, ale znasz ich pozycje
- Przetwarzasz dane w pętli po numerze wiersza

**W praktyce:** iloc używasz rzadziej niż loc — ale jest niezbędny
gdy pracujesz z pozycjami, nie nazwami.

In [ ]:
# Częsty błąd: iloc z nazwą kolumny
try:
    tips.iloc[0, 'total_bill']   # BŁĄD! iloc oczekuje numeru, nie nazwy
except ValueError as e:
    print(f"Błąd: {e}")
    print("iloc przyjmuje NUMER pozycji, nie nazwę!")
    print("Poprawnie: tips.iloc[0, 0] lub tips.loc[0, 'total_bill']")

---

## 2. loc — selekcja po etykiecie (nazwie)

`loc` = **l**abel **loc**ation — wybierasz po **nazwie** indeksu i kolumny.
To metoda, której będziecie używać **w 90% przypadków**.

### Kluczowa różnica loc vs iloc

| | iloc | loc |
|---|------|-----|
| **Klucz** | Numer pozycji (0, 1, 2...) | Etykieta (nazwa) |
| **Slice** | `0:3` → pozycje 0, 1, 2 | `'A':'C'` → A, B, C **włącznie!** |
| **Warunek** | Nie obsługuje | `df.loc[df['cena'] > 100]` ✓ |
| **Typowe użycie** | Pozycja (n-ty wiersz) | Filtrowanie, selekcja po nazwie |

**UWAGA na slice!** `iloc[0:3]` zwraca 3 elementy (0,1,2).
`loc['A':'C']` zwraca A, B, C — **włącznie z końcem!**

In [ ]:
# DataFrame z własnym indeksem — żeby loc miał sens
kraje = pd.DataFrame({
    'populacja_mln': [38.0, 83.2, 67.4, 47.4, 59.6],
    'pkb_mld_usd': [688, 4256, 2780, 1398, 2010],
    'bezrobocie_pct': [2.8, 3.0, 7.3, 11.7, 7.7]
}, index=['Polska', 'Niemcy', 'Francja', 'Hiszpania', 'Wlochy'])

print(kraje)

In [ ]:
# loc — wiersz po etykiecie
print("Polska:")
print(kraje.loc['Polska'])
print(f"\nPKB Polski: {kraje.loc['Polska', 'pkb_mld_usd']} mld USD")

In [ ]:
# loc — wiele etykiet + wybór kolumn
print("Polska i Niemcy — populacja i PKB:")
print(kraje.loc[['Polska', 'Niemcy'], ['populacja_mln', 'pkb_mld_usd']])

In [ ]:
# loc z warunkiem boolean — KLUCZOWE zastosowanie
print("Kraje z populacją powyżej 50 mln:")
print(kraje.loc[kraje['populacja_mln'] > 50])
print()
print("loc + warunek = filtrowanie. To jest serce analizy danych w Pandas.")

In [ ]:
# loc z warunkiem + wybór kolumn — precyzyjnie
print("Duże kraje — tylko populacja i bezrobocie:")
print(kraje.loc[kraje['populacja_mln'] > 50, ['populacja_mln', 'bezrobocie_pct']])
print()
print("loc[warunek, kolumny] — wybierasz WIERSZE warunkiem i KOLUMNY nazwą.")

### loc vs iloc — podsumowanie

```
# iloc — pozycja (numer)
tips.iloc[0]           # pierwszy wiersz
tips.iloc[0:5]         # wiersze 0-4
tips.iloc[:, 0]        # pierwsza kolumna

# loc — etykieta (nazwa/warunek)
kraje.loc['Polska']                 # wiersz 'Polska'
tips.loc[:, 'total_bill']           # kolumna 'total_bill'
tips.loc[tips['day'] == 'Sun']      # filtrowanie po warunku
```

**Prosta zasada: iloc = numer, loc = nazwa/warunek.**
W 90% przypadków używasz `loc` z warunkami lub `df[warunek]` (skrót).

> *"I learned very early the difference between knowing the name of something and knowing something."*
> *("Wcześnie zrozumiałem różnicę między znaniem nazwy czegoś a rozumieniem tego.")*
> — **Richard Feynman**, fizyk, noblista, legendarny wykładowca

Znać składnię `loc` i `iloc` to jedno. **Wiedzieć kiedy którego użyć** — to drugie.
Dlatego ćwiczymy na różnych przykładach, nie tylko czytamy dokumentację.

> **Ciekawostka:** Wes McKinney, twórca Pandas, przyznał w swoim blogu (2017),
> że jednym z jego największych żalów projektowych jest to, jak skomplikowane
> stało się indeksowanie w Pandas. Trzy sposoby (`[]`, `.loc`, `.iloc`) to wynik
> ewolucji biblioteki — każdy został dodany, żeby rozwiązać inny problem.
> Dlatego istnieje prosta zasada: **używaj `loc` do etykiet i warunków, `iloc` do pozycji**.
> Unikniesz 90% pułapek.

In [ ]:
# Szybkie porównanie: ten sam wynik, różna składnia
print("=== 5 najdroższych rachunków ===\n")

# Sposób 1: iloc (pozycja)
sorted_tips = tips.sort_values('total_bill', ascending=False)
print("iloc:", list(sorted_tips.iloc[0:5]['total_bill']))

# Sposób 2: loc + warunek
top5 = tips.loc[tips['total_bill'] >= 40]
print(f"loc:  {len(top5)} rachunków >= 40$")

# Sposób 3: nlargest (najprostsze)
print("nlargest:", list(tips.nlargest(5, 'total_bill')['total_bill']))

---

## 3. Filtrowanie z warunkami logicznymi

Filtrowanie to **serce pracy analityka**. Odpowiedź na pytanie:
*„pokaż mi tylko wiersze spełniające warunek"*.

### Analogia do Excela

| Co chcesz | Excel | Pandas |
|-----------|-------|--------|
| Rachunki > 30$ | Filtr automat. kolumna > 30 | `df[df['total_bill'] > 30]` |
| Sobota I drogo | Filtr: dzień=Sob ORAZ kwota>30 | `df[(df['day']=='Sat') & (df['total_bill']>30)]` |
| Sob LUB Niedz | Filtr zaawansowany | `df[df['day'].isin(['Sat','Sun'])]` |

Mechanizm jest identyczny jak filtrowanie boolean w NumPy (W03):
warunek tworzy maskę True/False → Pandas zwraca wiersze z True.

In [ ]:
# Jak działa filtrowanie — krok po kroku
# Krok 1: warunek tworzy Series True/False
maska = tips['total_bill'] > 30
print(f"Maska (pierwsze 10):")
print(maska.head(10))
print(f"\nTrue: {maska.sum()} wierszy z {len(maska)}")

In [ ]:
# Krok 2: maska jako indeks — zwraca tylko wiersze z True
drogie = tips[tips['total_bill'] > 30]
print(f"Rachunki > 30$: {len(drogie)} z {len(tips)}")
print(drogie.head())

### Operatory logiczne: AND, OR, NOT

| Operacja | Python (skalary) | Pandas (tablice) | Uwaga |
|----------|-----------------|-------------------|-------|
| I (AND) | `and` | `&` | **Nawiasy obowiązkowe!** |
| LUB (OR) | `or` | `\|` | **Nawiasy obowiązkowe!** |
| NIE (NOT) | `not` | `~` | Negacja maski |

**NAJCZĘSTSZY BŁĄD:** Brak nawiasów!
- Źle: `df[df['a'] > 5 & df['b'] < 10]` → `TypeError`
- Dobrze: `df[(df['a'] > 5) & (df['b'] < 10)]` → działa!

Dlaczego? Operator `&` ma wyższy priorytet niż `>` i `<`.
Bez nawiasów Python próbuje obliczyć `5 & df['b']` — stąd błąd.

In [ ]:
# AND — oba warunki spełnione (nawiasy obowiązkowe!)
sobota_drogie = tips[(tips['day'] == 'Sat') & (tips['total_bill'] > 30)]
print(f"Sobota + rachunek > 30$: {len(sobota_drogie)}")
print(sobota_drogie.head())

In [ ]:
# OR — przynajmniej jeden warunek
weekend = tips[(tips['day'] == 'Sat') | (tips['day'] == 'Sun')]
print(f"Weekend (Sob lub Niedz): {len(weekend)} z {len(tips)}")

In [ ]:
# NOT — negacja
niepalacze = tips[~(tips['smoker'] == 'Yes')]
print(f"Niepalacze (~Yes): {len(niepalacze)}")

# Prostsza alternatywa:
niepalacze2 = tips[tips['smoker'] == 'No']
print(f"Niepalacze (==No): {len(niepalacze2)}")
print("\nOba dają to samo. Negacja (~) przydaje się przy złożonych maskach.")

In [ ]:
# Praktyczny filtr: pytanie menedżera restauracji
# "Ile kobiet przychodzi w weekend na obiad?"
kobiety_weekend = tips[
    (tips['sex'] == 'Female') &
    (tips['day'].isin(['Sat', 'Sun'])) &
    (tips['time'] == 'Dinner')
]
print(f"Kobiety, weekend, obiad: {len(kobiety_weekend)} z {len(tips)}")
print(f"Średni rachunek: {kobiety_weekend['total_bill'].mean():.2f}$")

In [ ]:
# Złożony filtr — pytanie biznesowe
# "Duże grupy (4+), weekend, niepalacze, rachunek > 20$"
wynik = tips[
    (tips['size'] >= 4) &
    (tips['day'].isin(['Sat', 'Sun'])) &
    (tips['smoker'] == 'No') &
    (tips['total_bill'] > 20)
]
print(f"Znalezionych: {len(wynik)}")
print(wynik)
print()
print("Tip: każdy warunek w osobnej linii — czytelne i łatwe do modyfikacji.")

---

## 4. Skróty do częstych filtrów: isin, between, query

### isin() — czy wartość jest na liście

Zamiast pisać `(day == 'Sat') | (day == 'Sun') | (day == 'Fri')` — jedno wywołanie.

**Analogia Excel:** Funkcja LICZ.JEŻELI z listą wartości lub WYSZUKAJ.

In [ ]:
# isin — lista wartości
weekend = tips[tips['day'].isin(['Sat', 'Sun'])]
print(f"Weekend (isin): {len(weekend)}")

# Bez isin — dłuższe i mniej czytelne:
# tips[(tips['day'] == 'Sat') | (tips['day'] == 'Sun')]

# isin z negacją — wszystko POZA listą
dni_robocze = tips[~tips['day'].isin(['Sat', 'Sun'])]
print(f"Dni robocze (~isin): {len(dni_robocze)}")

In [ ]:
# isin — szczególnie przydatne przy wielu wartościach
# Bez isin (nieczytelne):
# tips[(tips['size'] == 2) | (tips['size'] == 3) | (tips['size'] == 4)]

# Z isin (czytelne):
srednie_grupy = tips[tips['size'].isin([2, 3, 4])]
print(f"Grupy 2-4 osoby: {len(srednie_grupy)} z {len(tips)}")
print(f"\nRozkład rozmiarów grup:")
print(srednie_grupy['size'].value_counts().sort_index())

### between() — zakres wartości

`between(min, max)` = `>= min` AND `<= max` — oba końce **włączone**.

**Analogia Excel:** Filtr niestandardowy z „od" i „do".

In [ ]:
# between — zakres wartości
srednie = tips[tips['total_bill'].between(15, 25)]
print(f"Rachunki 15-25$: {len(srednie)} ({len(srednie)/len(tips)*100:.0f}% wszystkich)")
print(srednie.head())
print()
print("between(15, 25) = (total_bill >= 15) & (total_bill <= 25)")

# Inne zakresy — porównanie
for lo, hi in [(10, 15), (15, 25), (25, 40), (40, 55)]:
    n = len(tips[tips['total_bill'].between(lo, hi)])
    print(f"  {lo:2d}-{hi:2d}$: {n:3d} rachunków ({n/len(tips)*100:.0f}%)")

### query() — filtrowanie stringiem (jak SQL)

`query()` pozwala napisać warunek jako tekst — czytelnie, bez powtarzania `df['...']`.

| Składnia standardowa | query() |
|---------------------|---------|
| `df[(df['a'] > 5) & (df['b'] == 'X')]` | `df.query("a > 5 and b == 'X'")` |

**Zalety:** czytelność, krótszy zapis. **Wady:** wolniejszy na małych danych, trudniejszy debug.

In [ ]:
# query — filtr SQL-owy
wynik = tips.query("total_bill > 30 and day == 'Sat' and smoker == 'No'")
print(f"Query: {len(wynik)}")
print(wynik.head())

# Porównanie z wersją standardową:
# tips[(tips['total_bill'] > 30) & (tips['day'] == 'Sat') & (tips['smoker'] == 'No')]
print("\nquery() jest czytelniejszy przy złożonych warunkach.")

In [ ]:
# query z zmiennymi zewnętrznymi (operator @)
prog = 20
wynik = tips.query("total_bill > @prog and size >= 4")
print(f"Rachunki > {prog}$ i grupa >= 4: {len(wynik)}")
print("\n@ pozwala użyć zmiennej Pythona wewnątrz query.")

### Porównanie 3 metod filtrowania

| Metoda | Składnia | Zalety | Wady |
|--------|---------|--------|------|
| **Boolean** | `df[(df['a']>5) & (df['b']=='X')]` | Uniwersalna, szybka | Powtarzanie `df['...']` |
| **query()** | `df.query("a > 5 and b == 'X'")` | Czytelna, krótka | Wolniejsza, trudniej debugować |
| **isin/between** | `df[df['a'].isin([1,2,3])]` | Wygodna dla list/zakresów | Tylko jeden warunek |

**Zasada:** Używaj tego, co jest **najczytelniejsze** w danym kontekście.
Proste filtry → boolean. Złożone → query. Listy wartości → isin.

---

## 5. Sortowanie i rankingi

Sortowanie to fundament raportowania — *„TOP-10 klientów"*, *„najsłabsze produkty"*,
*„rachunki od najdroższego"*.

### Analogia do Excela

| Co chcesz | Excel | Pandas |
|-----------|-------|--------|
| Sortuj A-Z | Dane → Sortuj A→Z | `df.sort_values('kol')` |
| Sortuj Z-A | Dane → Sortuj Z→A | `df.sort_values('kol', ascending=False)` |
| Top 5 | Sortuj + zaznacz 5 | `df.nlargest(5, 'kol')` |
| Sortuj po 2 kolumnach | Sortuj → Dodaj poziom | `df.sort_values(['kol1', 'kol2'])` |

In [ ]:
# Sortowanie po jednej kolumnie
print("TOP-5 rachunków (malejąco):")
print(tips.sort_values('total_bill', ascending=False).head())

In [ ]:
# Sortowanie po wielu kolumnach
print("Sortowanie: dzień (A-Z), potem rachunek (malejąco):")
print(tips.sort_values(['day', 'total_bill'], ascending=[True, False]).head(10))
print()
print("Lista kolumn + lista kierunków — każda kolumna może mieć inny kierunek.")

In [ ]:
# sort_values nie zmienia oryginału (chyba że inplace=True)
tips_sorted = tips.sort_values('total_bill', ascending=False)
print(f"Oryginał — pierwszy wiersz: total_bill = {tips.iloc[0]['total_bill']}")
print(f"Posortowany — pierwszy wiersz: total_bill = {tips_sorted.iloc[0]['total_bill']}")
print("\nOryginał nie zmieniony. sort_values zwraca KOPIĘ.")

In [ ]:
# Po sortowaniu indeks jest "pomieszany" — reset_index
sorted_tips = tips.sort_values('total_bill', ascending=False)
print("Indeks po sortowaniu:", list(sorted_tips.index[:5]))
print("→ Indeks zachowuje oryginalne numery wierszy")
print()

# reset_index() — nowy indeks od 0
clean = sorted_tips.reset_index(drop=True)
print("Po reset_index:", list(clean.index[:5]))
print("→ Czysty indeks 0, 1, 2, 3, 4")

### nlargest() i nsmallest() — TOP-N i BOTTOM-N

Szybszy i czytelniejszy sposób na znalezienie ekstremalnych wartości.

| Metoda | Odpowiednik | Kiedy używać |
|--------|-------------|-------------|
| `df.nlargest(5, 'kol')` | `sort_values(ascending=False).head(5)` | Top N |
| `df.nsmallest(5, 'kol')` | `sort_values(ascending=True).head(5)` | Bottom N |

In [ ]:
# Dodajmy kolumnę tip_pct — procent napiwku
tips['tip_pct'] = (tips['tip'] / tips['total_bill'] * 100).round(1)

# nlargest — TOP-5 napiwków procentowo
print("TOP-5 najhojniejszych (napiwek %):")
print(tips.nlargest(5, 'tip_pct')[['total_bill', 'tip', 'tip_pct', 'day']])

In [ ]:
# nsmallest — 5 najmniejszych rachunków
print("5 najtańszych rachunków:")
print(tips.nsmallest(5, 'total_bill')[['total_bill', 'tip', 'tip_pct', 'day', 'size']])

### Pattern: łączenie operacji (chaining)

W Pandas możesz łączyć operacje w łańcuch — każda metoda zwraca DataFrame/Series,
na którym wywołujesz następną metodę. To jak pipeline w Excelu: filtruj → sortuj → wybierz kolumny.

```python
wynik = (df
    .query("day == 'Sat'")       # filtruj
    .sort_values('tip', ascending=False)  # sortuj
    [['total_bill', 'tip']]      # kolumny
    .head(5))                    # top 5
```

Czytelne, zwięzłe, bez zmiennych tymczasowych.

In [ ]:
# Pattern: filtruj → sortuj → wyświetl
# "TOP-3 rachunki z soboty od niepalących"
(tips[(tips['day'] == 'Sat') & (tips['smoker'] == 'No')]
 .nlargest(3, 'total_bill')
 [['total_bill', 'tip', 'tip_pct', 'size']])

> **Ciekawostka:** W Stanach Zjednoczonych napiwek 15-20% jest normą społeczną
> — kelnerzy zarabiają poniżej płacy minimalnej i napiwki stanowią główną
> część ich dochodu. W Polsce napiwek to 5-10% i jest opcjonalny.
> Dlatego w datasecie  średni napiwek to ~16% — zupełnie inna kultura niż nasza.

> *"The first principle is that you must not fool yourself — and you are the easiest person to fool."*
> *("Pierwsza zasada: nie wolno ci oszukiwać samego siebie — a siebie oszukać najłatwiej.")*
> — **Richard Feynman**, *Surely You're Joking, Mr. Feynman!* (1985)

Dane bez kontekstu **kłamią**. 16% napiwku wygląda na mało — dopóki nie znasz kultury USA.
Gdy analizujecie dane, **zawsze pytajcie**: skąd pochodzą? kto je zbierał? w jakim kontekście?

---

## 6. Segmentacja klientów — zastosowanie biznesowe

Segmentacja to podział klientów/danych na grupy (segmenty) wg ustalonych reguł.
To jedno z najczęstszych zadań analityka — *„podziel klientów na VIP, Standard, Basic"*.

Narzędzia:
- `np.where()` — 2 grupy (tak/nie) — znacie z W04
- `np.select()` — 3+ grupy — znacie z W04
- `pd.cut()` — automatyczne przedziały (poznamy na W07)

In [ ]:
# Przypomnienie z W04: np.where (2 grupy) i np.select (3+ grup)
# np.where — jak IF w Excelu
tips['rozmiar_grupy'] = np.where(tips['size'] >= 4, 'Duza grupa', 'Mala grupa')

# np.select — jak zagnieżdżony IF w Excelu
tips['hojnosc'] = np.select(
    [tips['tip_pct'] > 20, tips['tip_pct'] > 10],
    ['Hojny', 'Przecietny'],
    default='Skapy'
)
print("Rozmiar grupy:", dict(tips['rozmiar_grupy'].value_counts()))
print("Hojność:      ", dict(tips['hojnosc'].value_counts()))

In [ ]:
# Analiza per segment — od ogółu do szczegółu
print("Statystyki per segment hojności:\n")
for segment in ['Hojny', 'Przecietny', 'Skapy']:
    grupa = tips[tips['hojnosc'] == segment]
    duze = grupa[grupa['size'] >= 4]
    print(f"{segment:12s}: n={len(grupa):3d}, "
          f"sr. rachunek={grupa['total_bill'].mean():5.1f}$, "
          f"sr. tip%={grupa['tip_pct'].mean():5.1f}%, "
          f"duze grupy={len(duze)/len(grupa)*100:.0f}%")

In [ ]:
# Segmentacja rachunków: Premium / Standard / Economy
tips['segment'] = np.select(
    [tips['total_bill'] > 30, tips['total_bill'] > 15],
    ['Premium', 'Standard'],
    default='Economy'
)
print(tips['segment'].value_counts())
print()
print("W prawdziwej firmie takie segmenty kierują strategią marketingu:")
print("  Premium → program lojalnościowy, VIP oferty")
print("  Standard → zachęty do zwiększenia wydatków")
print("  Economy → kupony rabatowe")

### Łączenie filtrowania z segmentacją

Segmenty + filtry = odpowiedzi na pytania biznesowe.

In [ ]:
# Pytanie: Czy hojni klienci częściej przychodzą w weekend?
for dzien in ['Sat', 'Sun', 'Thur', 'Fri']:
    hojni_dzien = tips[(tips['hojnosc'] == 'Hojny') & (tips['day'] == dzien)]
    total_dzien = tips[tips['day'] == dzien]
    pct = len(hojni_dzien) / len(total_dzien) * 100 if len(total_dzien) > 0 else 0
    print(f"{dzien:4s}: {pct:5.1f}% hojnych ({len(hojni_dzien)}/{len(total_dzien)})")

In [ ]:
# Szybki cross-tab: segment vs dzień
print("\nSegment hojności vs dzień tygodnia:")
print(pd.crosstab(tips['hojnosc'], tips['day']))
print()
print("crosstab — tabela krzyżowa. Jak tabela przestawna w Excelu!")

In [ ]:
# Wizualizacja — jeden wykres wart więcej niż tabela 244 wierszy
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 5))
kolory = {'Hojny': '#27ae60', 'Przecietny': '#2980b9', 'Skapy': '#c0392b'}
for seg, kolor in kolory.items():
    dane = tips[tips['hojnosc'] == seg]
    ax.scatter(dane['total_bill'], dane['tip'], c=kolor, label=seg, alpha=0.6, s=40)

ax.set_xlabel('Rachunek ($)')
ax.set_ylabel('Napiwek ($)')
ax.set_title('Napiwki vs rachunki — segmentacja klientów')
ax.legend(title='Segment')
plt.tight_layout()
plt.show()

print("Hojni (zieloni) = niskie rachunki, wysokie napiwki procentowo.")
print("Skąpi (czerwoni) = wyższe rachunki, niższe napiwki procentowo.")

> *"What I cannot create, I do not understand."*
> *("Czego nie potrafię stworzyć, tego nie rozumiem.")*
> — **Richard Feynman** (napis na tablicy w dniu jego śmierci, 1988)

Dlatego teraz **wy piszecie kod**. Czytanie przykładów to dopiero początek.

### Ćwiczenie A — Analiza danych HR

Nowy dataset, nowy kontekst. Spróbujcie sami — potem sprawdzimy razem.

In [ ]:
# Nowy dataset: pracownicy firmy technologicznej
pracownicy = pd.DataFrame({
    'imie': ['Anna K.', 'Piotr M.', 'Kasia W.', 'Tomek L.', 'Ola S.',
             'Marek D.', 'Ewa Z.', 'Jan N.', 'Zofia P.', 'Adam R.',
             'Iga T.', 'Paweł B.'],
    'dzial': ['Marketing', 'IT', 'Finanse', 'IT', 'HR',
              'Marketing', 'Finanse', 'IT', 'HR', 'Marketing',
              'IT', 'Finanse'],
    'pensja': [6500, 9200, 7800, 8500, 5800,
               7200, 8100, 11000, 5500, 6800,
               9800, 7500],
    'staz_lat': [3, 7, 5, 4, 2, 8, 6, 12, 1, 5, 3, 9],
    'ocena': [4.2, 3.8, 4.7, 4.1, 4.5, 3.5, 4.3, 4.8, 3.9, 4.0, 4.6, 3.7]
})
pracownicy.index = pracownicy['imie']
print(pracownicy[['dzial', 'pensja', 'staz_lat', 'ocena']])

In [ ]:
# Pytania HR — spróbuj sam! (3 minuty)
#
# 1. Kto zarabia powyżej 8000 zł? Wyświetl imię, dział i pensję.
#
# 2. Pracownicy IT z oceną >= 4.0 — ilu ich jest? Jaka średnia pensja?
#
# 3. TOP-3 najlepiej opłacani (nlargest) — kto to?
#
# 4. Pracownicy ze stażem 3-7 lat (between) — jaka średnia ocena?
#
# 5. Który dział ma najwyższą średnią ocenę?
#    (Podpowiedź: pętla for + filtrowanie, jak w sekcji segmentacji)


In [ ]:
# Rozwiązania — prowadzący pokazuje po 3 minutach
print("1. Pensja > 8000 zł:")
print(pracownicy.loc[pracownicy['pensja'] > 8000, ['dzial', 'pensja']])

print(f"\n2. IT z oceną >= 4.0:")
it_dobry = pracownicy[(pracownicy['dzial'] == 'IT') & (pracownicy['ocena'] >= 4.0)]
print(f"   {len(it_dobry)} osób, średnia pensja: {it_dobry['pensja'].mean():.0f} zł")

print(f"\n3. TOP-3 pensje:")
print(pracownicy.nlargest(3, 'pensja')[['dzial', 'pensja', 'ocena']])

print(f"\n4. Staż 3-7 lat:")
sr = pracownicy[pracownicy['staz_lat'].between(3, 7)]
print(f"   {len(sr)} osób, średnia ocena: {sr['ocena'].mean():.2f}")

print(f"\n5. Średnia ocena per dział:")
for dzial in pracownicy['dzial'].unique():
    gr = pracownicy[pracownicy['dzial'] == dzial]
    print(f"   {dzial:12s}: {gr['ocena'].mean():.2f}")

In [ ]:
# Bonus: segmentacja pracowników
pracownicy['poziom'] = np.select(
    [pracownicy['pensja'] > 9000, pracownicy['pensja'] > 7000],
    ['Senior', 'Mid'],
    default='Junior'
)
print("Segmenty pracowników:")
print(pracownicy[['dzial', 'pensja', 'ocena', 'poziom']])
print()
print("Takie segmentacje to codzienność w działach HR i controllingu.")

### Ćwiczenie B — Analiza rynku nieruchomości

Zmieniamy kontekst: z restauracji i HR na **rynek mieszkaniowy**.
Dane z portalu ogłoszeniowego — mieszkania na sprzedaż w Warszawie.

In [ ]:
# Dane: mieszkania na sprzedaż w Warszawie
mieszkania = pd.DataFrame({
    'dzielnica': ['Srodmiescie', 'Mokotow', 'Wola', 'Praga Poludnie', 'Ursynow',
                  'Zoliborz', 'Bemowo', 'Bielany', 'Targowek', 'Wilanow'],
    'metraz': [45, 68, 52, 38, 74, 55, 61, 43, 35, 92],
    'pokoje': [2, 3, 2, 1, 3, 2, 3, 2, 1, 4],
    'cena_tys': [680, 850, 520, 380, 590, 720, 480, 410, 290, 1150],
    'ocena_lokalizacji': [4.5, 4.7, 4.0, 3.8, 4.3, 4.8, 4.1, 3.9, 3.5, 4.6]
})
mieszkania.index = mieszkania['dzielnica']
print(mieszkania[['metraz', 'pokoje', 'cena_tys', 'ocena_lokalizacji']])

In [ ]:
# 1. Mieszkania w najlepszych lokalizacjach (ocena >= 4.5)
top_lokalizacje = mieszkania.loc[mieszkania['ocena_lokalizacji'] >= 4.5]
print("Najlepsze lokalizacje (ocena >= 4.5):")
print(top_lokalizacje[['cena_tys', 'metraz', 'ocena_lokalizacji']])

In [ ]:
# 2. Mieszkania poniżej 500 tys. z dobrą lokalizacją — "okazje"
okazje = mieszkania[(mieszkania['cena_tys'] < 500) & (mieszkania['ocena_lokalizacji'] > 4.0)]
print(f"Okazje (tanio + dobra lokalizacja): {len(okazje)}")
print(okazje[['metraz', 'cena_tys', 'ocena_lokalizacji']])

In [ ]:
# 2b. Ranking mieszkań wg ceny za metr kwadratowy
mieszkania['cena_m2'] = (mieszkania['cena_tys'] / mieszkania['metraz'] * 1000).round(0)
print("Ranking: cena za m² (od najtańszego):")
print(mieszkania.sort_values('cena_m2')[['metraz', 'cena_tys', 'cena_m2']])

In [ ]:
# 3. TOP-3 najdroższe mieszkania
top3 = mieszkania.nlargest(3, 'cena_tys')
print("TOP-3 najdroższe mieszkania:")
print(top3[['metraz', 'pokoje', 'cena_tys', 'ocena_lokalizacji']])

### Ćwiczenie C — Segmentacja mieszkań

Podziel mieszkania na segmenty cenowe i odpowiedz:
1. Użyj `np.select` do stworzenia kolumny `segment`: Premium (>800 tys.), Standard (>450 tys.), Budget
2. Ile mieszkań w każdym segmencie?
3. Jaki średni metraż i ocena lokalizacji per segment?

In [ ]:
# Segmentacja mieszkań wg ceny
mieszkania['segment'] = np.select(
    [mieszkania['cena_tys'] > 800, mieszkania['cena_tys'] > 450],
    ['Premium', 'Standard'],
    default='Budget'
)
print(mieszkania[['cena_tys', 'segment']])
print()
print(f"Segmenty: {mieszkania['segment'].value_counts().to_dict()}")

In [ ]:
# Statystyki per segment cenowy
for seg in ['Premium', 'Standard', 'Budget']:
    grupa = mieszkania[mieszkania['segment'] == seg]
    print(f"{seg:10s}: {len(grupa)} mieszk., "
          f"sr. metraz={grupa['metraz'].mean():.0f} m², "
          f"sr. ocena={grupa['ocena_lokalizacji'].mean():.1f}, "
          f"sr. cena={grupa['cena_tys'].mean():.0f} tys. zl")

### Bonus: .assign() — tworzenie kolumn w chainie

Metoda `.assign()` pozwala tworzyć nowe kolumny w łańcuchu operacji —
bez modyfikowania oryginalnego DataFrame.

In [ ]:
# assign — nowa kolumna w chainie (nie modyfikuje oryginału)
wynik = (tips
    .assign(tip_pln=tips['tip'] * 4.0)  # nowa kolumna: napiwek w PLN
    .query("tip_pln > 20")              # filtruj
    .nlargest(5, 'tip_pln')             # top 5
    [['total_bill', 'tip', 'tip_pln', 'day']]
)
print("TOP-5 napiwków w PLN (kurs 4.0):")
print(wynik)

---

## 7. Mini-ćwiczenia na wykładzie

Trzy krótkie zadania — sprawdzenie, czy omówione metody są zrozumiałe.
Czas: ~10 minut.

### Ćwiczenie A — iloc i loc

Korzystając z datasetu `tips`, wyświetl:
1. Wiersz nr 100 (`iloc`)
2. Wiersze 50-55, kolumny 0-1 (`iloc`)
3. Co 20-ty wiersz (`iloc`)

In [ ]:
# Ćwiczenie A — uzupełnij luki:

# 1. Wiersz nr 100:
# tips.iloc[___]

# 2. Wiersze 50-55, kolumny 0-1:
# tips.iloc[___:___, ___:___]

# 3. Co 20-ty wiersz:
# tips.iloc[::___]

### Ćwiczenie B — filtrowanie z warunkami

Odpowiedz na pytania, łącząc warunki (`&`, `|`, `isin`, `between`):

In [ ]:
# Ćwiczenie B — uzupełnij luki:

# 1. Kobiety-palaczki z weekendu:
# tips[(tips['sex'] == ___) & (tips['smoker'] == ___) & (tips['day'].isin([___, ___]))]

# 2. Rachunki 20-30$ od niepalących mężczyzn — średni napiwek:
# dane = tips[(tips['total_bill'].between(___, ___)) & (tips['smoker'] == ___) & (tips['sex'] == ___)]
# print(f"Średni napiwek: {dane['tip'].mean():.2f}$")

### Ćwiczenie C — łańcuch operacji (chaining)

Napisz **jeden łańcuch operacji**, który odpowie na pytanie:
*„5 największych rachunków od grup 2-osobowych w piątek lub sobotę"*

In [ ]:
# Ćwiczenie C — uzupełnij łańcuch operacji:

# wynik = (tips
#     [tips['size'] == ___]                    # grupy 2-osobowe
#     [tips['day'].isin([___, ___])]           # piątek lub sobota
#     .nlargest(___, '___')                    # 5 największych rachunków
#     [['total_bill', 'tip', 'day', 'size']]   # kolumny
# )
# print(wynik)

---

## Podgląd laboratorium

Na laboratorium przećwiczycie to wszystko na datasecie tips:

**Start — 3 komendy:**
```
cd C:\Users\student\python2
.venv\Scripts\Activate.ps1
code .
```

**Ćwiczenie 1:** loc i iloc — selekcja wierszy i kolumn, własny indeks (20 min)

**Ćwiczenie 2:** Filtrowanie — AND, OR, isin, between, query (20 min)

**Ćwiczenie 3:** Samodzielna analiza — rankingi, porównania dzień po dniu, pytania biznesowe, anomalie (30 min)

**Ćwiczenie 4:** Segmentacja klientów + commit na GitHub (15 min)

---

## Podsumowanie — ściąga

### Selekcja

| Metoda | Co robi | Przykład |
|--------|---------|---------|
| `df.iloc[w, k]` | Selekcja po pozycji | `df.iloc[0:5, 0:3]` |
| `df.loc[w, k]` | Selekcja po etykiecie | `df.loc['Anna', 'cena']` |
| `df.loc[warunek]` | Filtrowanie | `df.loc[df['cena'] > 100]` |
| `df[warunek]` | Filtrowanie (skrót) | `df[df['cena'] > 100]` |
| `df[['a', 'b']]` | Wybór kolumn | `df[['cena', 'nazwa']]` |

### Filtrowanie

| Operacja | Składnia | Uwaga |
|----------|---------|-------|
| AND | `(w1) & (w2)` | Nawiasy obowiązkowe! |
| OR | `(w1) \| (w2)` | Nawiasy obowiązkowe! |
| NOT | `~(warunek)` | Negacja maski |
| Lista wartości | `df['kol'].isin(['a','b'])` | Zastępuje OR |
| Zakres | `df['kol'].between(10, 20)` | Oba końce włączone |
| Query | `df.query("a > 5 and b == 'X'")` | Styl SQL |

### Sortowanie

| Metoda | Co robi | Przykład |
|--------|---------|---------|
| `sort_values('kol')` | Sortuj rosnąco | `df.sort_values('cena')` |
| `sort_values(ascending=False)` | Sortuj malejąco | `df.sort_values('cena', ascending=False)` |
| `sort_values(['a','b'])` | Sortuj po wielu | `df.sort_values(['dzien','kwota'], ascending=[True,False])` |
| `nlargest(n, 'kol')` | Top N | `df.nlargest(5, 'sprzedaz')` |
| `nsmallest(n, 'kol')` | Bottom N | `df.nsmallest(5, 'cena')` |

### Segmentacja

| Metoda | Kiedy | Przykład |
|--------|-------|---------|
| `np.where(w, tak, nie)` | 2 grupy | `np.where(x>100, 'Drogi', 'Tani')` |
| `np.select([w1,w2], [v1,v2], default)` | 3+ grupy | System ocen, segmenty klientów |

---

## Źródła

| Źródło | Opis |
|--------|------|
| [Pandas — Indexing and Selecting](https://pandas.pydata.org/docs/user_guide/indexing.html) | Oficjalna dokumentacja loc/iloc |
| [Pandas — Boolean Indexing](https://pandas.pydata.org/docs/user_guide/indexing.html#boolean-indexing) | Filtrowanie warunkami |
| [Pandas — sort_values](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.sort_values.html) | Sortowanie |
| [Pandas — query](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.query.html) | Filtrowanie SQL-owe |
| [Pandas — isin](https://pandas.pydata.org/docs/reference/api/pandas.Series.isin.html) | Lista wartości |
| [Real Python — Pandas Indexing](https://realpython.com/pandas-dataframe/#accessing-and-modifying-data) | Praktyczny poradnik |

---

## Na koniec

> *"If you can't explain it simply, you don't understand it well enough."*
> *("Jeśli nie potrafisz tego wyjaśnić prosto, nie rozumiesz tego wystarczająco dobrze.")*
> — przypisywane Albertowi Einsteinowi

### Pytanie na zakończenie

**Zastanówcie się (30 sekund):** Która z dzisiejszych metod wydaje Wam się **najużyteczniejsza
w przyszłej pracy analityka**? `loc` z warunkami? `query()`? `nlargest()`? Segmentacja?

Nie ma złej odpowiedzi — ważne, żebyście wiedzieli **po co** sięgnąć, gdy dostaniecie
prawdziwy dataset od szefa.

---

**Za tydzień: czyszczenie danych** — brakujące wartości (NaN), duplikaty, błędne typy,
literówki w danych. W prawdziwym życiu 60-80% czasu analityka to czyszczenie —
więc to będzie ważny wykład.

**Zadanie domowe (nieoceniane):** Jesteś analitykiem w oceanarium.
Dyrektor prosi o raport z datasetu `penguins` (seaborn):
(1) Ile pingwinów gatunku Chinstrap? (2) Jakie 3 wyspy mają pingwiny?
(3) Pingwiny z wyspy Dream — jaka średnia masa? (4) TOP-3 najdłuższe dzioby.
(5) Ile samic waży powyżej 4000g? Notebook na GitHub.